# EYE — AI talking-avatar test video

First end-to-end try: script -> voice (Edge-TTS, free) -> avatar image (SDXL-Turbo) -> lip-synced talking head (SadTalker) -> final vertical MP4 with a watermark.

Run cells top to bottom. Runtime -> Change runtime type -> **T4 GPU** first.

**Known fix baked in**: cell 1 force-reinstalls `numpy<1.24` after SadTalker's own install — SadTalker's deps reference `np.VisibleDeprecationWarning`, removed in numpy>=1.24 (Colab's default). If you already hit `AttributeError: module 'numpy' has no attribute 'VisibleDeprecationWarning'` in an old session: **Runtime -> Restart session**, then run all cells top to bottom again (a numpy reinstall doesn't take effect for code already imported in that kernel).

This is still a **first try, not a finished pipeline** — if a different cell errors, paste it back and we fix it.

In [ ]:
# 1) Install dependencies
!pip install -q edge-tts diffusers transformers accelerate safetensors
!git clone -q https://github.com/OpenTalker/SadTalker.git
%cd SadTalker
!pip install -q -r requirements.txt
!bash scripts/download_models.sh
%cd /content

# SadTalker's pinned deps expect an older numpy (some still reference
# np.VisibleDeprecationWarning, removed in numpy>=1.24). Colab's default
# numpy is newer, so pin it down AFTER the other installs so nothing bumps
# it back up. Requires a runtime restart to take effect (see markdown above).
!pip install -q 'numpy<1.24' --force-reinstall

In [ ]:
# 2) Hugging Face login (SDXL-Turbo is a gated model — accept the license at
# https://huggingface.co/stabilityai/sdxl-turbo first, then paste a read token here)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# 3) Generate the avatar portrait (SDXL-Turbo, 2 steps, a few seconds on a T4)
import torch
from diffusers import AutoPipelineForText2Image

pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16"
).to("cuda")

# Edit this prompt to change the character. Keep it a front-facing portrait —
# SadTalker needs a clear, mostly-frontal face to animate well.
avatar_prompt = (
    "friendly cartoon marketing mentor character, front-facing portrait, "
    "sitting at a laptop, warm smile, simple clean background, vector illustration style"
)
image = pipe(prompt=avatar_prompt, num_inference_steps=2, guidance_scale=0.0).images[0]
image.save("/content/avatar.png")
image

In [ ]:
# 4) Script + voice (Edge-TTS — free, no key). Swap the script text for your own topic.
script_text = (
    "Struggling with a high bounce rate? Here are three quick fixes. "
    "One: speed up your load time, every extra second costs you visitors. "
    "Two: make your headline crystal clear in the first five seconds. "
    "Three: add one obvious call to action above the fold. "
    "Try these today and watch people stay longer on your site."
)

import edge_tts

# Voice options: https://gist.github.com/BettyJJ/17cbaa1de96235a7f5773b8690a20462
communicate = edge_tts.Communicate(script_text, voice="en-US-GuyNeural")
await communicate.save("/content/voice.mp3")

In [ ]:
# 5) Lip-sync the avatar to the voice (SadTalker). This is the slow step —
# a few minutes on a T4 for a ~20s clip.
%cd /content/SadTalker
!python inference.py \
  --driven_audio /content/voice.mp3 \
  --source_image /content/avatar.png \
  --result_dir /content/results \
  --still --preprocess full --enhancer gfpgan
%cd /content

In [ ]:
# 6) Assemble final vertical (9:16) MP4 with a watermark, via ffmpeg (preinstalled on Colab)
import glob

raw_video = sorted(glob.glob("/content/results/*/*.mp4"))[-1]
print("SadTalker output:", raw_video)

!ffmpeg -y -i "{raw_video}" \
  -vf "scale=1080:1920:force_original_aspect_ratio=decrease,pad=1080:1920:(ow-iw)/2:(oh-ih)/2:color=black,drawtext=text='EYE Analytics':fontcolor=white:fontsize=36:x=20:y=20" \
  -c:a copy /content/final_test.mp4

print("Done: /content/final_test.mp4")

In [ ]:
# 7) Preview + download
from IPython.display import Video
display(Video("/content/final_test.mp4", embed=True, width=360))

from google.colab import files
files.download("/content/final_test.mp4")